## Depedências

In [1]:
import kagglehub
import os
import keras
import numpy as np
import pandas as pd
from PIL import Image
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Dense, Dropout, Activation, Flatten, Concatenate, Add, Softmax
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Input, BatchNormalization
from tensorflow.keras.initializers import he_normal
from tensorflow.keras import optimizers
from tensorflow.keras.callbacks import LearningRateScheduler, TensorBoard, CSVLogger
from tensorflow.keras.utils import get_file
from tensorflow.keras import backend as K
from tensorflow.keras.utils import to_categorical
from tensorflow.keras import optimizers
from tensorflow.keras.utils import plot_model
from tensorflow.keras.models import load_model
from tensorflow.keras.callbacks import CSVLogger
from tensorflow.keras.utils import set_random_seed
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from scipy import stats
from sklearn.model_selection import train_test_split
from matplotlib import pyplot as plt

# Relations for Fashion MNIST
c2_to_c1 = {
    0:0, 1:0, 2:0,
    3:1, 4:1, 5:1,
    6:2, 7:2, 8:2,
    9:3, 10:3, 11:3,
}
fine_to_c2 = {
    0:0, 1:0, 2:0,
    3:1, 4:1, 5:1,
    6:2, 7:2, 8:2,
    9:3,
    10:4, 11:4, 12:4,
    13:5, 14:5, 15:5,
    16:6, 17:6, 18:6,
    19:7,
    20:8, 21:8, 22:8,
    23:9, 24:9, 25:9,
    26:10, 27:10, 28:10,
    29:11
}

unique_fine = [*range(0, 30, 1)]
unique_c2 = np.vectorize(fine_to_c2.get)(unique_fine)
unique_c1 = np.vectorize(c2_to_c1.get)(unique_c2)

relations = []
for i in range(30):
  relations.append([unique_c1[i],unique_c2[i]+2,unique_fine[i]+8])

# Computes hierarchical metrics
def hierarchical_metrics(true,pred):
  true_labels = []
  true_fine = true[2].argmax(axis=1)+8
  true_c2 = true[1].argmax(axis=1)+2
  true_c1 = true[0].argmax(axis=1)
  for i in range(len(true_fine)):
    true_labels.append([true_c1[i],true_c2[i],true_fine[i]])
  pred_labels = []
  pred_c1 = pred[0].argmax(axis = 1)
  pred_c2 = pred[1].argmax(axis = 1)+2
  pred_fine = pred[2].argmax(axis = 1)+8
  for i in range(len(pred_c1)):
    pred_labels.append([pred_c1[i],pred_c2[i],pred_fine[i]])
  preci = precision(true_labels,pred_labels)
  reca = recall(true_labels,pred_labels)
  f_1 = f1(true_labels,pred_labels)

  consistent_examples = 1
  correct_pred = 0
  test_set_size = len(true_labels)
  for i in range(test_set_size):
    if [pred_c1[i],pred_c2[i],pred_fine[i]] in relations:
        consistent_examples = consistent_examples + 1
    if [pred_c1[i],pred_c2[i],pred_fine[i]] == true_labels[i]:
        correct_pred = correct_pred +1
  h_accuracy = correct_pred/test_set_size
  h_consistency = (consistent_examples-1)/test_set_size

  return h_accuracy,h_consistency,f_1

# Hierarchical metrics, proposed by Kiritchenko et al (2005)
# Implementation
# https://gitlab.com/dacs-hpi/hiclass/-/blob/main/hiclass/metrics.py


def precision(y_true: np.ndarray, y_pred: np.ndarray):
    """
    Compute precision score for hierarchical classification.

    hP = sum(|S intersection T|) / sum(|S|),
    where S is the set consisting of the most specific class(es) predicted for a test example and all respective ancestors
    and T is the set consisting of the true most specific class(es) for a test example and all respective ancestors.

    Parameters
    ----------
    y_true : np.array of shape (n_samples, n_levels)
        Ground truth (correct) labels.
    y_pred : np.array of shape (n_samples, n_levels)
        Predicted labels, as returned by a classifier.
    Returns
    -------
    precision : float
        What proportion of positive identifications was actually correct?
    """
    assert len(y_true) == len(y_pred)
    sum_intersection = 0
    sum_prediction_and_ancestors = 0
    for ground_truth, prediction in zip(y_true, y_pred):
        sum_intersection = sum_intersection + len(
            set(ground_truth).intersection(set(prediction))
        )
        sum_prediction_and_ancestors = sum_prediction_and_ancestors + len(
            set(prediction)
        )
    precision = sum_intersection / sum_prediction_and_ancestors
    return precision


def recall(y_true: np.ndarray, y_pred: np.ndarray):
    """
    Compute recall score for hierarchical classification.

    hR = sum(|S intersection T|) / sum(|T|),
    where S is the set consisting of the most specific class(es) predicted for a test example and all respective ancestors
    and T is the set consisting of the true most specific class(es) for a test example and all respective ancestors.

    Parameters
    ----------
    y_true : np.array of shape (n_samples, n_levels)
        Ground truth (correct) labels.
    y_pred : np.array of shape (n_samples, n_levels)
        Predicted labels, as returned by a classifier.
    Returns
    -------
    recall : float
        What proportion of actual positives was identified correctly?
    """
    assert len(y_true) == len(y_pred)
    sum_intersection = 0
    sum_prediction_and_ancestors = 0
    for ground_truth, prediction in zip(y_true, y_pred):
        sum_intersection = sum_intersection + len(
            set(ground_truth).intersection(set(prediction))
        )
        sum_prediction_and_ancestors = sum_prediction_and_ancestors + len(
            set(ground_truth)
        )
    recall = sum_intersection / sum_prediction_and_ancestors
    return recall


def f1(y_true: np.ndarray, y_pred: np.ndarray):
    """
    Compute f1 score for hierarchical classification.

    hF = 2 * hP * hR / (hP + hR),
    where hP is the hierarchical precision and hR is the hierarchical recall.

    Parameters
    ----------
    y_true : np.array of shape (n_samples, n_levels)
        Ground truth (correct) labels.
    y_pred : np.array of shape (n_samples, n_levels)
        Predicted labels, as returned by a classifier.
    Returns
    -------
    f1 : float
        Weighted average of the precision and recall
    """
    assert len(y_true) == len(y_pred)
    prec = precision(y_true, y_pred)
    rec = recall(y_true, y_pred)
    f1 = 2 * prec * rec / (prec + rec)
    return f1

## Configurações Gerais

In [2]:
# Baixa última versão do dataset
path = kagglehub.dataset_download("paramaggarwal/fashion-product-images-dataset")

print("Path to dataset files:", path)

Path to dataset files: /kaggle/input/fashion-product-images-dataset


In [3]:
# Carrega o arquivo de metadados
styles_csv_path = '/kaggle/input/fashion-product-images-dataset/fashion-dataset/styles.csv'
styles_df = pd.read_csv(styles_csv_path, on_bad_lines='skip')

#----------------------------Primeira filtragem pelo masterCategory----------------------------
# Conta as ocorrências da categoria master
master_category_counts = styles_df['masterCategory'].value_counts()

# Pega as 4 maiores
top_4_master_categories = master_category_counts.nlargest(4).index.tolist()

# Filtra pelas 4 maiores
filtered_styles_df = styles_df[styles_df['masterCategory'].isin(top_4_master_categories)]
#----------------------------Primeira filtragem----------------------------

#----------------------------Segunda filtragem pelo subCategory----------------------------
top_3_subcategories_per_master = {}
filtered_styles_df_by_top_subcategories = pd.DataFrame()

for master_cat in top_4_master_categories:
    # Filtra o dataframe por categoria principal
    df_master_cat = styles_df[styles_df['masterCategory'] == master_cat]

    # Obtenha as 3 subcategorias mais frequentes para esta categoria principal
    top_sub_categories = df_master_cat['subCategory'].value_counts().nlargest(3).index.tolist()
    top_3_subcategories_per_master[master_cat] = top_sub_categories

    # Filtra pelas 3 subcategorias
    df_filtered_sub = df_master_cat[df_master_cat['subCategory'].isin(top_sub_categories)]

    filtered_styles_df_by_top_subcategories = pd.concat([filtered_styles_df_by_top_subcategories, df_filtered_sub])

#----------------------------Segunda filtragem pelo subCategory----------------------------

#----------------------------Terceira filtragem----------------------------
top_3_article_types_per_subcategory = {}
dataset = pd.DataFrame()

# Iterate through the subcategories that are already filtered
for sub_cat in filtered_styles_df_by_top_subcategories['subCategory'].unique():
    df_sub_cat = filtered_styles_df_by_top_subcategories[filtered_styles_df_by_top_subcategories['subCategory'] == sub_cat]

    top_article_types = df_sub_cat['articleType'].value_counts().nlargest(3).index.tolist()
    top_3_article_types_per_subcategory[sub_cat] = top_article_types

    df_filtered_article = df_sub_cat[df_sub_cat['articleType'].isin(top_article_types)]

    dataset = pd.concat([dataset, df_filtered_article])
#----------------------------Terceira filtragem----------------------------

In [4]:
## Dimensões da imagem
height, width = 32, 32

channel = 3
if K.image_data_format() == 'channels_first':
    input_shape = (channel, height, width)
else:
    input_shape = (height, width, channel)

In [5]:
train_size = int(dataset.shape[0] * 0.70)
test_size = int(np.floor(dataset.shape[0] * 0.15))
val_size = int(np.ceil(dataset.shape[0] * 0.15))

train_df, temp_df = train_test_split(
    dataset,
    test_size=(val_size + test_size) / len(dataset),
    random_state=42,
    stratify=dataset['articleType']
)

val_df, test_df = train_test_split(
    temp_df,
    test_size=test_size / (val_size + test_size),
    random_state=42,
    stratify=temp_df['articleType']
)


# Quantidade de classes no primeiro nível hierárquico camada
coarse1_classes = 4

# Quantidade de classes no segundo nível hierárquico camada
coarse2_classes = 12

# Quantidade de classes no último nível hierárquico (folhas da árvore)
num_classes  = 83

batch_size   = 128
epochs       = 60

In [6]:
def scheduler(epoch):
  learning_rate_init = 0.001
  if epoch > 55:
    learning_rate_init = 0.0002
  if epoch > 70:
    learning_rate_init = 0.00005
  return learning_rate_init

In [7]:
class LossWeightsModifier(keras.callbacks.Callback):
  def __init__(self, alpha, beta, gamma):
    self.alpha = alpha
    self.beta = beta
    self.gamma = gamma
    # customize your behavior
  def on_epoch_end(self, epoch, logs={}):
    if epoch == 15:
      K.set_value(self.alpha, 0.1)
      K.set_value(self.beta, 0.8)
      K.set_value(self.gamma, 0.1)
    if epoch == 25:
      K.set_value(self.alpha, 0.1)
      K.set_value(self.beta, 0.2)
      K.set_value(self.gamma, 0.7)
    if epoch == 35:
      K.set_value(self.alpha, 0)
      K.set_value(self.beta, 0)
      K.set_value(self.gamma, 1)

In [8]:
#----------Pega os pesos do modelo VGG16 pré-treinado--------
WEIGHTS_PATH = 'https://github.com/fchollet/deep-learning-models/releases/download/v0.1/vgg16_weights_tf_dim_ordering_tf_kernels.h5'
weights_path = get_file(
    'vgg16_weights_tf_dim_ordering_tf_kernels.h5',
    WEIGHTS_PATH,
    cache_subdir='models',
)

# Carregando os dados
train_path = '/kaggle/input/fashion-product-images-dataset/fashion-dataset/images/'

x_train = []
y_train =[]

x_test = []
y_test = []

x_val = []
y_val = []

id_to_index = {image_id: index for index, image_id in enumerate(dataset['id'])}

for image_id in dataset['id']:
    image_filename = str(image_id) + '.jpg'
    image_path = os.path.join(train_path, image_filename)

    if os.path.exists(image_path):
      dataset_index = id_to_index[image_id]

      if image_id in train_df['id'].values:
        img = Image.open(image_path)
        img = img.resize((width, height))
        img_array = np.array(img)
        x_train.append(img_array)
        y_train.append(dataset.iloc[dataset_index]['articleType'])

      if image_id in test_df['id'].values:
        img = Image.open(image_path)
        img = img.resize((width, height))
        img_array = np.array(img)
        x_test.append(img_array)
        y_test.append(dataset.iloc[dataset_index]['articleType'])


      if image_id in val_df['id'].values:
        img = Image.open(image_path)
        img = img.resize((width, height))
        img_array = np.array(img)
        x_val.append(img_array)
        y_val.append(dataset.iloc[dataset_index]['articleType'])

# Normalizando os dados
x_train = np.reshape(x_train, (train_size, channel, height, width)).transpose(0, 2, 3, 1).astype("float32")
x_train = (x_train-np.mean(x_train)) / np.std(x_train)

x_test = np.reshape(x_train, (test_size, channel, height, width)).transpose(0, 2, 3, 1).astype("float32")
x_test = (x_test-np.mean(x_test)) / np.std(x_test)

x_val = np.reshape(x_train, (val_size, channel, height, width)).transpose(0, 2, 3, 1).astype("float32")
x_val = (x_val-np.mean(x_val)) / np.std(x_val)

553467096/553467096 ━━━━━━━━━━━━━━━━━━━━ 17s 0us/step


ValueError: cannot reshape array of size 67445760 into shape (21957,3,32,32)